# KOF Governors: Notebook Explicado

Este notebook toma la base del **KOF ETH Zurich** sobre gobernadores de bancos centrales y la transforma en una tabla limpia.

## Qué problema resuelve

El archivo Excel no viene en formato tabular normal:

- cada **columna** representa un país
- la fila 0 contiene el **ISO3**
- la fila 1 contiene el **nombre del país**
- las filas siguientes contienen texto libre con uno o varios gobernadores

Por eso no basta con leer el Excel y exportarlo. Hay que:

1. leer la hoja correcta
2. entender qué país corresponde a cada columna
3. parsear cada celda
4. extraer nombre y años
5. construir una tabla limpia


## 1. Imports y configuración

Aquí cargamos librerías básicas y definimos el archivo de entrada, la hoja y el nombre del CSV de salida.

In [1]:
import csv
import re
from pathlib import Path

import openpyxl


INPUT_FILE = Path("cbg_turnover_v23upload.xlsx")
SHEET_NAME = "governors v2023"
OUTPUT_FILE = Path("kof_governors.csv")

## 2. Funciones auxiliares

Estas funciones hacen el trabajo fino:

- `extract_year`: saca un año desde una cadena como `03-1995` o `1995`
- `clean_name`: limpia títulos y notas como `(interim)`
- `parse_cell`: toma una celda completa y trata de convertirla en uno o más registros estructurados

In [2]:
def extract_year(s):
    """Extrae el año de una cadena. Devuelve 'En cargo' si dice present/current."""
    if not s:
        return ""
    s = str(s).lower().strip()
    if any(x in s for x in ["present", "current", "ongoing"]):
        return "En cargo"
    m = re.search(r"\d{4}", s)
    return m.group() if m else ""


def clean_name(name):
    """Limpia el nombre: elimina títulos y notas entre paréntesis."""
    name = re.sub(
        r"^(dr\.|mr\.|mrs\.|ms\.|prof\.|ing\.|sir\s)\s*",
        "",
        str(name),
        flags=re.IGNORECASE,
    )
    name = re.sub(
        r"\s*\((?:reapp|first time|interim|acting|second|1st|2nd|3rd|reappointed)[^)]*\)",
        "",
        name,
        flags=re.IGNORECASE,
    )
    name = re.sub(r"\s+", " ", name).strip().rstrip(".,;")
    return name


def parse_cell(text, country_name, iso):
    """
    Parsea una celda que puede contener uno o varios gobernadores.
    Devuelve una lista de registros estructurados.
    """
    if not text or str(text).strip() in ["None", ""]:
        return []

    text = str(text).strip()

    skip_keywords = [
        "established", "est. in", "est.in", "code:", "from 1985 onwards member",
        "member of beac", "(beac)", "(bceao)", "(eccb)", "no central bank",
        "code: -999", "exists since", "est. 1", "est.1",
    ]
    if any(p in text.lower() for p in skip_keywords):
        return []

    entries = re.split(r"\n", text)
    results = []

    patterns = [
        r"(\d{1,2}-\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{1,2}-\d{4})\s+(.+)",
        r"(\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{4})\s+(.+)",
        r"(\d{1,2}-\d{4})\s+to\s+(present|current)\s+(.+)",
        r"(\d{4})\s+to\s+(\d{4})\s+(.+)",
        r"(\d{4})\s+to\s+(present|current)\s+(.+)",
        r"(\d{4})\s+-\s+(\d{4})\s+(.+)",
        r"(\d{4})\s+-\s+(present|current)\s+(.+)",
        r"(.+?)\s+(\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{4})$",
        r"(.+?)\s+(\d{4})\s+to\s+(\d{4})$",
    ]

    for entry in entries:
        entry = entry.strip()
        if not entry:
            continue

        for i, pat in enumerate(patterns):
            m = re.search(pat, entry, re.IGNORECASE)
            if not m:
                continue

            if i < 7:
                start_raw, end_raw, name = m.group(1), m.group(2), m.group(3)
            else:
                name, start_raw, end_raw = m.group(1), m.group(2), m.group(3)

            name = clean_name(name)
            if len(name) < 2:
                continue

            start_year = extract_year(start_raw)
            end_year = extract_year(end_raw)
            if not start_year:
                continue

            results.append(
                {
                    "iso3": iso,
                    "country": country_name,
                    "name": name,
                    "position": "President / Governor",
                    "start_year": start_year,
                    "end_year": end_year,
                    "status": "Actual" if end_year == "En cargo" else "Histórico",
                }
            )
            break

    return results

## 3. Cargar el Excel

Leemos el archivo y convertimos la hoja en una lista de filas.

Esto nos deja ver claramente la estructura:

- `rows[0]`: ISO3
- `rows[1]`: nombres de países
- `rows[2:]`: datos

In [3]:
wb = openpyxl.load_workbook(INPUT_FILE, read_only=True)
ws = wb[SHEET_NAME]
rows = list(ws.iter_rows(values_only=True))

print(f"Total filas leídas: {len(rows)}")
print("Primeras 5 columnas de la fila ISO:")
print(rows[0][:5])
print("Primeras 5 columnas de la fila país:")
print(rows[1][:5])

FileNotFoundError: [Errno 2] No such file or directory: 'cbg_turnover_v23upload.xlsx'

## 4. Construir el mapa de países

Como cada columna representa un país, primero construimos un diccionario:

- clave: índice de columna
- valor: `iso` y `name`

Eso permite luego saber qué país corresponde a cada celda.

In [ ]:
iso_codes = rows[0]
country_names = rows[1]

countries = {}
for i, (iso, name) in enumerate(zip(iso_codes, country_names)):
    if iso and name:
        countries[i] = {
            "iso": str(iso).strip(),
            "name": str(name).strip(),
        }

print(f"Países encontrados: {len(countries)}")
list(countries.items())[:5]

## 5. Parsear todas las celdas

Ahora sí recorremos todas las filas de datos y todas las columnas.

Para cada celda válida:

1. identificamos el país de la columna
2. tomamos el texto de la celda
3. llamamos a `parse_cell(...)`
4. acumulamos los registros obtenidos

In [ ]:
all_records = []

for row in rows[2:]:
    for col_idx, val in enumerate(row):
        if col_idx not in countries:
            continue
        if not val or str(val).strip() in ["None", ""]:
            continue

        ci = countries[col_idx]
        parsed = parse_cell(str(val), ci["name"], ci["iso"])
        all_records.extend(parsed)

print(f"Registros parseados: {len(all_records)}")
all_records[:5]

## 6. Deduplicar registros

La base puede traer repeticiones. Aquí usamos como clave:

- `iso3`
- `name.lower()`
- `start_year`

Si aparece la misma combinación más de una vez, nos quedamos solo con una.

In [ ]:
seen = set()
unique = []

for r in all_records:
    key = (r["iso3"], r["name"].lower(), r["start_year"])
    if key not in seen:
        seen.add(key)
        unique.append(r)

print(f"Registros únicos: {len(unique)}")
unique[:5]

## 7. Ordenar y exportar a CSV

Ordenamos por país y año de inicio, y luego escribimos el CSV final.

In [ ]:
unique.sort(key=lambda x: (x["country"], x["start_year"]))

fieldnames = ["iso3", "country", "name", "position", "start_year", "end_year", "status"]

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(unique)

print(f"CSV guardado en: {OUTPUT_FILE}")

## 8. Resumen estadístico

Terminamos con un resumen rápido para validar el resultado.

In [ ]:
paises = len(set(r["iso3"] for r in unique))
actuales = sum(1 for r in unique if r["end_year"] == "En cargo")
historicos = sum(1 for r in unique if r["status"] == "Histórico")

print("─" * 40)
print(f"Total registros : {len(unique)}")
print(f"Países únicos   : {paises}")
print(f"Actuales        : {actuales}")
print(f"Históricos      : {historicos}")
print("─" * 40)

## 9. Qué hace bien este enfoque

- entiende una estructura Excel rara
- limpia nombres
- detecta varios formatos de fecha
- descarta notas institucionales
- exporta un CSV usable

## 10. Limitaciones

- depende de patrones de texto fijos
- siempre asigna la posición `President / Governor`
- puede perder casos si aparece un formato nuevo no contemplado
- la deduplicación puede ser agresiva en algunos homónimos
